# Capability: lookup

The `lookup` capability resolves a field value by searching known datasets (lookup tables). This is useful for mapping codes to names — for example, converting a supplier ID to a full supplier name.

In [ ]:
from pydantic import BaseModel

import paxman
import paxman.contract.adapters.pydantic  # noqa: F401


class SupplierInfo(BaseModel):
    supplier_id: str
    supplier_name: str


sample = "Supplier ID: ACME-01"

result = paxman.normalize(sample, SupplierInfo)
print(f"Data: {result.normalized_data}")

## Register a custom capability

Paxman's `register_capability` lets you add custom extraction logic. Let's build a simple supplier-code lookup.

In [ ]:
from paxman import register_capability
from paxman.capabilities.spec import CapabilitySpec, CapabilityTier
from paxman.capabilities.base import CapabilityContext
from paxman.capabilities.result import CapabilityResult, Candidate, EvidenceRef
from paxman.types import ConfidenceBand

LOOKUP = {
    "ACME-01": "ACME Hardware Supplies",
    "GLOBEX-99": "Globex Construction",
}


class SupplierLookup:
    @property
    def spec(self):
        return CapabilitySpec(
            name="supplier_lookup",
            description="Resolves supplier codes to full names",
            tier=CapabilityTier.TIER_1,
            input_types=["STRING"],
            output_type="STRING",
        )

    def invoke(self, ctx: CapabilityContext):
        input_str = ctx.input_data if isinstance(ctx.input_data, str) else str(ctx.input_data)
        candidates = []
        for code, name in LOOKUP.items():
            if code in input_str:
                candidates.append(Candidate(value=name, confidence=ConfidenceBand.HIGH, evidence_ref=EvidenceRef(path="lookup")))
        return CapabilityResult(candidates=candidates)


register_capability(SupplierLookup())
print("Registered custom supplier_lookup capability.")

In [ ]:
result2 = paxman.normalize(sample, SupplierInfo)
print(f"Data with lookup: {result2.normalized_data}")

## How it works

1. The V1 `lookup` capability is a generic framework — you provide the lookup table or function.
2. Custom capabilities are registered via `paxman.register_capability(capability_instance)`.
3. The **planner** considers all registered capabilities when building the field plan.
4. **Tier-1** capabilities (including lookup) are preferred over tier-2 text extraction.

> **Reference:** See `docs/howto/add_capability.md` for the full capability authoring guide.

## Try it yourself

- Build a lookup table for product SKU to product name and register it as a capability.
- What happens if the lookup returns no candidates? Check the unresolved fields.
- Register two capabilities that match the same field — how does the reconciler choose?